# Insurance Linear Regression

Start from the beginning and write the code yourself.


In [46]:
import pandas as pd
import numpy as np

In [47]:
insurance = pd.read_csv("data/insurance.csv")
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [48]:
print(insurance.shape)
print()
insurance.info()

(1338, 7)

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [49]:
print(insurance.isnull().sum())
print()
print("Duplicates:", insurance.duplicated().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicates: 1


In [50]:
insurance[["bmi", "charges"]].describe()

,bmi,charges
count,1338.000000,1338.000000
mean,30.663397,13270.422265
std,6.098187,12110.011237
min,15.960000,1121.873900
25%,26.296250,4740.287150
50%,30.400000,9382.033000
75%,34.693750,16639.912515
max,53.130000,63770.428010


In [51]:
insurance = insurance.drop_duplicates()
insurance.duplicated().sum()

np.int64(0)

In [52]:
Q1 = insurance["charges"].quantile(0.25)
Q3 = insurance["charges"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower)
print("Upper bound:", upper)

Q1: 4746.344
Q3: 16657.71745
IQR: 11911.37345
Lower bound: -13120.716174999998
Upper bound: 34524.777625


In [53]:
outliers = insurance[(insurance["charges"] < lower) | (insurance["charges"] > upper)]
outliers.shape

(139, 7)

In [54]:
insurance_clean = insurance[
    (insurance["charges"] >= lower) &
    (insurance["charges"] <= upper)
]

print("Before:", insurance.shape)
print("After:", insurance_clean.shape)

Before: (1337, 7)
After: (1198, 7)


In [55]:
insurance_clean

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [56]:
X = insurance_clean[["bmi"]]
y = insurance_clean["charges"]

print(X.head())
print()
print(y.head())

      bmi
0  27.900
1  33.770
2  33.000
3  22.705
4  28.880

0    16884.92400
1     1725.55230
2     4449.46200
3    21984.47061
4     3866.85520
Name: charges, dtype: float64


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(958, 1)
(240, 1)
(958,)
(240,)


In [58]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[-59.71]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['bmi']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.161e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,1


In [59]:
y_pred = model.predict(X_test)
y_pred[:10]

array([ 9919.03718209,  9590.03544381,  9941.1298578 , 10111.90027004,
       10177.58119783,  9816.93319435, 10117.57271381,  9822.60563811,
        9334.77547446,  9244.01637425])

In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)
print("R2 Percentage:", round(r2 * 100, 2), "%")

MAE: 5652.792755308421
MSE: 53592293.544948384
R2 Score: 0.0037210864792750487
R2 Percentage: 0.37 %


In [61]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results.head(10)

,Actual,Predicted
1315,11272.33139,9919.037182
1134,19673.33573,9590.035444
117,19107.77960,9941.129858
492,2196.47320,10111.900270
69,17663.14420,10177.581198
887,5272.17580,9816.933194
367,8017.06115,10117.572714
1181,2850.68375,9822.605638
462,15230.32405,9334.775474
960,2730.10785,9244.016374


In [62]:
insurance_clean
X = insurance_clean[["age","bmi","children","charges"]].corr()
X

,age,bmi,children,charges
age,1.000000,0.119704,0.039201,0.436891
bmi,0.119704,1.000000,0.002798,-0.066453
children,0.039201,0.002798,1.000000,0.082932
charges,0.436891,-0.066453,0.082932,1.000000


In [63]:
insurance_clean.groupby("smoker")["charges"].mean()

smoker
no      8362.048001
yes    22014.245543
Name: charges, dtype: float64

In [64]:
insurance_clean["smoker_encoded"] = insurance_clean["smoker"].map({"no": 0, "yes": 1})
insurance_clean[["smoker", "smoker_encoded", "charges"]].head()

,smoker,smoker_encoded,charges
0,yes,1,16884.92400
1,no,0,1725.55230
2,no,0,4449.46200
3,no,0,21984.47061
4,no,0,3866.85520


In [65]:
X = insurance_clean[["age","bmi","children","smoker_encoded"]]
y = insurance_clean["charges"]

In [66]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](4,)","[ 244.87, 60.45, 350.91,15076.38]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](4,)","['age','bmi','children','smoker_encoded']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-3577
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,4


In [67]:
y_pred = model.predict(X_test)
y_pred[:10]

array([ 2892.98325581,  5323.85992507, 20640.6015445 ,  2346.829407  ,
       20858.18834619,  7053.15073659,  8919.87057571,  4108.92975013,
       14609.78244164,  3821.19889213])

In [68]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)
print("R2 Percentage:", round(r2 * 100, 2), "%")

MAE: 2596.1601248249594
MSE: 23053222.09480877
R2 Score: 0.5714413856442805
R2 Percentage: 57.14 %


In [73]:
insurance_encoded = pd.get_dummies(insurance_clean, drop_first=True)
insurance_encoded = insurance_encoded.drop(columns=["smoker_encoded"])
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,False,True,False,False,True
1,18,33.770,1,1725.55230,True,False,False,True,False
2,28,33.000,3,4449.46200,True,False,False,True,False
3,33,22.705,0,21984.47061,True,False,True,False,False
4,32,28.880,0,3866.85520,True,False,True,False,False


In [82]:
X = insurance_encoded.drop(columns=["charges"])
y = insurance_encoded["charges"]

print(X.head())

   age     bmi  children  sex_male  smoker_yes  region_northwest  \
0   19  27.900         0     False        True             False   
1   18  33.770         1      True       False             False   
2   28  33.000         3      True       False             False   
3   33  22.705         0      True       False              True   
4   32  28.880         0      True       False              True   

   region_southeast  region_southwest  
0             False              True  
1              True             False  
2              True             False  
3             False             False  
4             False             False  


In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ 243.21, 80.65, 354.34,..., -300.93,-1110.97,-1338.77]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['age','bmi','children',...,'region_northwest','region_southeast', 'region_southwest']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-3189
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,8


In [86]:
y_pred = model.predict(X_test)
y_pred[:10]

array([ 3346.67710886,  5567.86348931, 20435.19105984,  3211.35755787,
       20101.73923309,  7686.55181239,  9448.36621264,  4760.4265601 ,
       15670.78313169,  4680.27993714])

In [87]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)
print("R2 Percentage:", round(r2 * 100, 2), "%")

MAE: 2609.191164667132
MSE: 22880829.736783497
R2 Score: 0.5746461537143137
R2 Percentage: 57.46 %


## Conclusion

In this project, I used the Insurance dataset to predict medical insurance charges with Linear Regression.

The dataset had 1338 rows and 7 columns. I checked the data and found no missing values, but there was 1 duplicate row. I removed the duplicate before training the model.

I also checked outliers in the `charges` column using the IQR method. After removing extreme outliers, the dataset was reduced from 1337 rows to 1198 rows.

First, I trained a simple model using only `bmi` as the feature. This model performed very poorly:

- R² Score: 0.37%
- MAE: 5652.79
- MSE: 53592293.54

Then I checked the relationship between the features and `charges`. The correlation table showed that `age` had a stronger relationship with `charges` than `bmi`. I also found that smoker status was very important because smokers had much higher average charges than non-smokers.

After that, I improved the model by adding more features:

- age
- bmi
- children
- smoker_encoded

This improved the model a lot:

- R² Score: 57.14%
- MAE: 2596.16
- MSE: 23053222.09

Finally, I used `pd.get_dummies()` to encode the categorical columns like `sex`, `smoker`, and `region`. The final model used all numeric and encoded features. The final result was:

- R² Score: 57.46%
- MAE: 2609.19
- MSE: 22880829.74

Overall, the model improved a lot after adding important features, especially smoker status. The final model explains about 57% of the variation in insurance charges.
